In [4]:
import polars as pl
import math

DATA_DIR = "../data"
NUM_QUOTES = 3

def quote_horizontal(agg_fn, side, field, n=3):
    return agg_fn([
        pl.col(f"{side}_quote_{i}").struct.field(f"{side}_{field}_{i}")
        for i in range(1, n + 1)
    ])
    
min_bid = quote_horizontal(pl.min_horizontal, "bid", "price", NUM_QUOTES)
bid_vol = quote_horizontal(pl.sum_horizontal, "bid", "volume", NUM_QUOTES)
max_ask = quote_horizontal(pl.max_horizontal, "ask", "price", NUM_QUOTES)
ask_vol = quote_horizontal(pl.sum_horizontal, "ask", "volume", NUM_QUOTES)

cleaned_data = (
    pl.read_csv(DATA_DIR + "/round-0/day-1/prices_round_0_day_-1.csv", separator=";")
    .drop(["day", "profit_and_loss"])
    .with_columns([
        pl.struct([f"bid_price_{i}", f"bid_volume_{i}"]).alias(f"bid_quote_{i}")
        for i in range(1, 4)
    ] + [
        pl.struct([f"ask_price_{i}", f"ask_volume_{i}"]).alias(f"ask_quote_{i}")
        for i in range(1, 4)
    ])
    .with_columns(
        (max_ask - min_bid).alias("spread") # change to ask_price_1 - bid_price_1
    )
    .with_columns(
       ((bid_vol - ask_vol) / (bid_vol + ask_vol)).alias("imbalance")
    )
    .drop([f"bid_price_{i}" for i in range(1, 4)] +
          [f"bid_volume_{i}" for i in range(1, 4)] +
          [f"ask_price_{i}" for i in range(1, 4)] +
          [f"ask_volume_{i}" for i in range(1, 4)])
    .select(["timestamp", "product", "bid_quote_1", "bid_quote_2", "bid_quote_3",
             "ask_quote_1", "ask_quote_2", "ask_quote_3", "mid_price", "spread", "imbalance"])
)
split_cleaned_data = cleaned_data.partition_by("product", as_dict=True)
split_cleaned_data = {k[0]: v for k, v in split_cleaned_data.items()}

In [5]:
tomatoes_data = split_cleaned_data['TOMATOES']
tomatoes_data = tomatoes_data.with_columns(
    (pl.col("mid_price") / pl.col("mid_price").shift(1)).log(base=math.e).alias("log_returns")
).with_columns(
    (pl.col("log_returns")).shift(1).alias("future_log_returns") # this is showing past returns instead of future returns, change to -1
).with_columns(
    (pl.corr("imbalance", "future_log_returns")).alias("imbalance/future_log_ret_corr") # do bucket analysis instead of scalar correlation metric
).with_columns(
    (pl.col("bid_quote_1").struct.field("bid_volume_1") + pl.col("ask_quote_1").struct.field("ask_volume_1")).alias("depth")
)
tomatoes_data

timestamp,product,bid_quote_1,bid_quote_2,bid_quote_3,ask_quote_1,ask_quote_2,ask_quote_3,mid_price,spread,imbalance,log_returns,future_log_returns,imbalance/future_log_ret_corr,depth
i64,str,struct[2],struct[2],struct[2],struct[2],struct[2],struct[2],f64,i64,f64,f64,f64,f64,i64
0,"""TOMATOES""","{4999,5}","{4998,15}","{null,null}","{5013,5}","{5014,15}","{null,null}",5006.0,16,0.0,null,null,0.020165,10
100,"""TOMATOES""","{5000,8}","{4998,21}","{null,null}","{5013,8}","{5014,21}","{null,null}",5006.5,16,0.0,0.0001,null,0.020165,16
200,"""TOMATOES""","{5000,10}","{4999,20}","{null,null}","{5013,10}","{5015,20}","{null,null}",5006.5,16,0.0,0.0,0.0001,0.020165,20
300,"""TOMATOES""","{5000,9}","{4999,21}","{null,null}","{5014,9}","{5015,21}","{null,null}",5007.0,16,0.0,0.0001,0.0,0.020165,18
400,"""TOMATOES""","{5000,5}","{4999,20}","{null,null}","{5014,5}","{5015,20}","{null,null}",5007.0,16,0.0,0.0,0.0001,0.020165,10
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
999500,"""TOMATOES""","{4955,5}","{4950,10}","{4948,23}","{4963,10}","{4964,23}","{null,null}",4959.0,16,0.070423,0.000605,-0.000101,0.020165,15
999600,"""TOMATOES""","{4955,5}","{4949,6}","{4948,18}","{4963,6}","{4964,18}","{null,null}",4959.0,16,0.09434,0.0,0.000605,0.020165,11
999700,"""TOMATOES""","{4950,10}","{4948,16}","{null,null}","{4963,10}","{4964,16}","{null,null}",4956.5,16,0.0,-0.000504,0.0,0.020165,20
